### Bernoulli Naive Bayes

In [ ]:
import numpy as np

class BernoulliNaiveBayes:
    def __init__(self, alpha=1.0): # alpha for Laplace smoothing
        self.alpha = alpha
        self.priors = {}
        self.feature_probs = {}
        self.classes = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)

        for c in self.classes:
            X_c = X[y == c] # Samples belonging to class c
            n_c = X_c.shape[0] # Number of samples in class c

            # Calculate class prior P(C_k)
            self.priors[c] = (n_c + self.alpha) / (n_samples + len(self.classes) * self.alpha)

            # Calculate feature probabilities P(F_i | C_k)
            # Use Laplace smoothing: (count(F_i=1 and C_k) + alpha) / (count(C_k) + 2*alpha)
            self.feature_probs[c] = {}
            for i in range(n_features):
                # P(F_i=1 | C_k)
                p_feature_given_class = (np.sum(X_c[:, i]) + self.alpha) / (n_c + 2 * self.alpha)
                self.feature_probs[c][i] = {
                    1: p_feature_given_class,
                    0: 1 - p_feature_given_class
                }

    def predict(self, X):
        predictions = []
        for x in X:
            posteriors = []
            for c in self.classes:
                # Start with log of prior
                log_posterior = np.log(self.priors[c])

                # Add log likelihoods
                for i, feature_val in enumerate(x):
                    log_posterior += np.log(self.feature_probs[c][i][feature_val])
                posteriors.append(log_posterior)
            predictions.append(self.classes[np.argmax(posteriors)])
        return np.array(predictions)

print("BernoulliNaiveBayes class defined.")

BernoulliNaiveBayes class defined.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generate synthetic binary data
# We'll create a dataset with 100 samples, 5 binary features
# and two classes (0 and 1). The features will have different
# probabilities of being 1 for each class.

np.random.seed(42)
n_samples = 200
n_features = 5

# Class 0: Features have low probability of being 1
X0 = np.random.rand(n_samples // 2, n_features)
X0 = (X0 < 0.2).astype(int) # 20% chance of being 1
y0 = np.zeros(n_samples // 2, dtype=int)

# Class 1: Features have higher probability of being 1
X1 = np.random.rand(n_samples // 2, n_features)
X1 = (X1 > 0.7).astype(int) # 30% chance of being 1 (1-0.7)
y1 = np.ones(n_samples // 2, dtype=int)

X_bnb = np.vstack((X0, X1))
y_bnb = np.hstack((y0, y1))

# Shuffle the data
shuffle_idx = np.random.permutation(n_samples)
X_bnb = X_bnb[shuffle_idx]
y_bnb = y_bnb[shuffle_idx]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_bnb, y_bnb, test_size=0.3, random_state=42)

print(f"Generated binary data: {X_bnb.shape[0]} samples, {X_bnb.shape[1]} features.")
print(f"First 5 samples of X_bnb:\n{X_bnb[:5]}")
print(f"First 5 labels of y_bnb: {y_bnb[:5]}\n")

# Create an instance of BernoulliNaiveBayes and fit the data
bnb_model = BernoulliNaiveBayes(alpha=1.0) # Using Laplace smoothing with alpha=1.0
bnb_model.fit(X_train, y_train)
print("BernoulliNaiveBayes model trained.")

# Make predictions on the test set
y_pred_bnb = bnb_model.predict(X_test)

# Evaluate accuracy
accuracy_bnb = accuracy_score(y_test, y_pred_bnb)
print(f"Bernoulli Naive Bayes Test Accuracy: {accuracy_bnb:.2f}")

Generated binary data: 200 samples, 5 features.
First 5 samples of X_bnb:
[[0 1 0 0 1]
 [0 1 0 1 0]
 [0 0 0 0 0]
 [0 0 0 0 1]
 [0 0 0 0 1]]
First 5 labels of y_bnb: [1 1 0 0 0]

BernoulliNaiveBayes model trained.
Bernoulli Naive Bayes Test Accuracy: 0.47


The low test accuracy (0.47) for the Bernoulli Naive Bayes model is primarily due to the way the synthetic data was generated, making the two classes very difficult to distinguish based on their features.

Here's why:

**Non-Discriminative Features:**
- For **Class 0**, features were set to be 1 with a probability of 0.2 (i.e., P(feature=1 | class=0) = 0.2).
- For **Class 1**, features were set to be 1 with a probability of 0.3 (i.e., P(feature=1 | class=1) = 0.3).

    The difference between these probabilities (0.3 - 0.2 = 0.1) is very small. This means that for any given feature, its value (0 or 1) doesn't strongly indicate which class the sample belongs to. Both classes have a similar tendency for their features to be 0 or 1.

**Close to Random Chance:**
- With two classes, a random guess would yield an accuracy of 0.50 (50%).
- An accuracy of 0.47 is slightly *worse* than random guessing. This indicates that the small statistical differences between the classes in the generated data are either insufficient for the model to learn a strong pattern, or in some cases, might even lead to slightly counter-intuitive predictions when the signals are so weak.

Essentially, the data itself does not contain strong enough signals in its features to clearly separate the two classes, making it challenging for any classifier, including Bernoulli Naive Bayes, to achieve high accuracy.

### Bernoulli Naive Bayes V2

In [20]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from time import time
import nltk
nltk.download('stopwords') # Moved this line here to ensure stopwords are downloaded before use.
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer

from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

sns.set_style("whitegrid")
tokenizer = RegexpTokenizer(r'\w+')
stop_words = set(stopwords.words('english'))
stop_words.update(['s','t','m','1','2'])

class naive_bayes:
    def __init__(self, K, D):
        self.K = K #number of classes
        self.D = D #dictionary size

        self.pi = np.ones(K) #class priors
        self.theta = np.ones((self.D, self.K)) #bernoulli parameters

    def fit(self, X_train, y_train):

        num_docs = X_train.shape[0]
        for doc in range(num_docs):

            label = y_train[doc]
            self.pi[label] += 1

            for word in range(self.D):
                if (X_train[doc][word] > 0):
                    self.theta[word][label] += 1
                #end if
            #end for
        #end for

        #normalize pi and theta
        self.pi = self.pi/np.sum(self.pi)
        self.theta = self.theta/np.sum(self.theta, axis=0)

    def predict(self, X_test):

        num_docs = X_test.shape[0]
        logp = np.zeros((num_docs,self.K))
        for doc in range(num_docs):
            for kk in range(self.K):
                logp[doc][kk] = np.log(self.pi[kk])
                for word in range(self.D):
                    if (X_test[doc][word] > 0):
                        logp[doc][kk] += np.log(self.theta[word][kk])
                    else:
                        logp[doc][kk] += np.log(1-self.theta[word][kk])
                    #end if
                #end for
            #end for
        #end for
        return np.argmax(logp, axis=1)

if __name__ == "__main__":

    # The nltk.download('stopwords') call is now moved to the top of the script.

    #load data
    print("loading 20 newsgroups dataset...")
    tic = time()
    classes = ['sci.space', 'comp.graphics', 'rec.autos', 'rec.sport.hockey']
    dataset = fetch_20newsgroups(shuffle=True, random_state=0, remove=('headers','footers','quotes'), categories=classes)
    X_train, X_test, y_train, y_test = train_test_split(dataset.data, dataset.target, test_size=0.5, random_state=0)
    toc = time()
    print("elapsed time: %.4f sec" %(toc - tic))
    print("number of training docs: ", len(X_train))
    print("number of test docs: ", len(X_test))

    print("vectorizing input data...")
    # Convert stop_words set to a list as required by CountVectorizer
    cnt_vec = CountVectorizer(tokenizer=tokenizer.tokenize, analyzer='word', ngram_range=(1,1), max_df=0.8, min_df=2, max_features=1000, stop_words=list(stop_words))
    cnt_vec.fit(X_train)
    toc = time()
    print("elapsed time: %.2f sec" %(toc - tic))
    vocab = cnt_vec.vocabulary_
    idx2word = {val: key for (key, val) in vocab.items()}
    print("vocab size: ", len(vocab))

    X_train_vec = cnt_vec.transform(X_train).toarray()
    X_test_vec = cnt_vec.transform(X_test).toarray()

    print("naive bayes model MLE inference...")
    K = len(set(y_train)) #number of classes
    D = len(vocab) #dictionary size
    nb_clf = naive_bayes(K, D)
    nb_clf.fit(X_train_vec, y_train)

    print("naive bayes prediction...")
    y_pred = nb_clf.predict(X_test_vec)
    nb_clf_acc = accuracy_score(y_test, y_pred)
    print("test set accuracy: ", nb_clf_acc)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


loading 20 newsgroups dataset...
elapsed time: 1.7966 sec
number of training docs:  1185
number of test docs:  1186
vectorizing input data...
elapsed time: 1.98 sec
vocab size:  1000


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


naive bayes model MLE inference...
naive bayes prediction...
test set accuracy:  0.821247892074199
